# 00 — Exploración del Dataset de Demanda Eléctrica

Este notebook es el punto de partida de nuestro proyecto MLOps. Vamos a explorar un **dataset sintético de demanda eléctrica horaria** que simula el consumo de energía de una ciudad mediana durante 3 años.

## Objetivos
- Entender la estructura y características del dataset
- Detectar valores nulos, duplicados y outliers
- Visualizar la serie temporal y sus estacionalidades
- Identificar qué features tienen mayor relación con la variable objetivo

## Variable objetivo
`demand_kwh`: consumo eléctrico en kilovatios-hora por hora

> **Nota:** El dataset es generado sintéticamente con patrones realistas: ciclos diarios, semanales y anuales, efectos de temperatura, días festivos y ruido aleatorio.

In [ ]:
# Instalación de dependencias
# Si estás en Google Colab, ejecuta esta celda primero
%pip install pandas numpy plotly scikit-learn --quiet

## 1. Generación del Dataset

El dataset simula la demanda eléctrica horaria de una ciudad mediana. Los patrones incluidos son:

- **Ciclo diario**: consumo bajo de madrugada, picos a las 8-9h y 19-21h
- **Ciclo semanal**: menor consumo los fines de semana
- **Ciclo anual**: mayor consumo en invierno (calefacción) y verano (aire acondicionado)
- **Temperatura**: correlación directa con el consumo
- **Festivos**: reducción del ~15% en días festivos
- **Ruido**: variabilidad aleatoria realista

**Features generados:**
| Feature | Tipo | Descripción |
|---------|------|-------------|
| `timestamp` | datetime | Fecha y hora (resolución horaria) |
| `hour` | int | Hora del día (0-23) |
| `day_of_week` | int | Día de la semana (0=Lunes) |
| `month` | int | Mes del año (1-12) |
| `is_weekend` | int | 1 si es fin de semana |
| `is_holiday` | int | 1 si es festivo |
| `temperature_c` | float | Temperatura en grados Celsius |
| `demand_kwh` | float | **Variable objetivo** |


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

# ─── Parámetros del dataset ────────────────────────────────────────────────
START_DATE = '2021-01-01'
END_DATE   = '2023-12-31 23:00:00'
FREQ       = 'h'
BASE_DEMAND = 500.0   # kWh base

# ─── Lista de festivos nacionales (España) ─────────────────────────────────
HOLIDAYS = [
    '2021-01-01', '2021-01-06', '2021-04-02', '2021-05-01',
    '2021-10-12', '2021-11-01', '2021-12-06', '2021-12-08', '2021-12-25',
    '2022-01-01', '2022-01-06', '2022-04-15', '2022-05-01',
    '2022-10-12', '2022-11-01', '2022-12-06', '2022-12-08', '2022-12-25',
    '2023-01-01', '2023-01-06', '2023-04-07', '2023-05-01',
    '2023-10-12', '2023-11-01', '2023-12-06', '2023-12-08', '2023-12-25',
]

# ─── Generación del índice temporal ───────────────────────────────────────
timestamps = pd.date_range(start=START_DATE, end=END_DATE, freq=FREQ)
n = len(timestamps)
print(f'Número de registros horarios: {n:,}')
print(f'Período: {timestamps[0]} → {timestamps[-1]}')

# ─── Ciclo diario (perfil de carga típico) ─────────────────────────────────
hour_profile = np.array([
    0.60, 0.55, 0.52, 0.50, 0.52, 0.58,  # 00-05: madrugada
    0.70, 0.85, 0.95, 0.92, 0.88, 0.87,  # 06-11: mañana
    0.85, 0.83, 0.82, 0.83, 0.86, 0.92,  # 12-17: tarde temprana
    0.98, 1.00, 0.97, 0.92, 0.83, 0.72   # 18-23: tarde-noche (pico vespertino)
])

hours       = timestamps.hour.values
dow         = timestamps.dayofweek.values
months      = timestamps.month.values
date_strs   = timestamps.strftime('%Y-%m-%d')

# ─── Componentes de la señal ───────────────────────────────────────────────
daily_component   = hour_profile[hours]                          # [0.5 – 1.0]
weekly_component  = np.where(dow >= 5, 0.80, 1.0)               # fin de semana -20%
annual_component  = 1.0 + 0.20 * np.cos(2 * np.pi * (months - 1) / 12)  # invierno/verano
holiday_component = np.where(np.isin(date_strs, HOLIDAYS), 0.85, 1.0)

# ─── Temperatura (correlacionada con ciclo anual) ──────────────────────────
temp_annual = 12.0 + 10.0 * np.sin(2 * np.pi * (timestamps.dayofyear - 80) / 365)
temp_daily  = 3.0  * np.sin(2 * np.pi * (hours - 6) / 24)
temp_noise  = np.random.normal(0, 2, n)
temperature = temp_annual + temp_daily + temp_noise

# Demanda adicional por temperatura extrema (frío o calor)
temp_effect = 0.003 * (temperature - 18.0) ** 2   # mínimo en 18°C

# ─── Demanda final ─────────────────────────────────────────────────────────
noise  = np.random.normal(0, 15, n)   # ruido gaussiano
demand = (
    BASE_DEMAND
    * daily_component
    * weekly_component
    * annual_component
    * holiday_component
    * (1 + temp_effect)
    + noise
)
demand = np.clip(demand, 100, 1500)   # límites físicos

# ─── DataFrame ────────────────────────────────────────────────────────────
df = pd.DataFrame({
    'timestamp':    timestamps,
    'hour':         hours.astype(int),
    'day_of_week':  dow.astype(int),
    'month':        months.astype(int),
    'is_weekend':   (dow >= 5).astype(int),
    'is_holiday':   np.isin(date_strs, HOLIDAYS).astype(int),
    'temperature_c': np.round(temperature, 2),
    'demand_kwh':   np.round(demand, 2),
})

print(f'\nDataFrame generado exitosamente: {df.shape}')

## 2. Vista general del dataset

Vamos a hacer una primera inspección del DataFrame: sus primeras filas, dimensiones, tipos de datos y estadísticas descriptivas.

In [ ]:
print('=== Primeras filas ===')
display(df.head(10))

print(f'\n=== Dimensiones ===' )
print(f'Filas: {df.shape[0]:,}   |   Columnas: {df.shape[1]}')

print('\n=== Tipos de datos ===')
display(df.dtypes.rename('dtype').to_frame())

print('\n=== Estadísticas descriptivas ===')
display(df.describe().round(2))

## 3. Análisis de valores nulos y duplicados

Antes de cualquier análisis, verificamos la **integridad del dataset**:
- ¿Hay valores nulos que puedan sesgar los modelos?
- ¿Hay timestamps duplicados que rompan la serie temporal?
- ¿La serie temporal es continua (sin huecos)?

In [ ]:
# ─── Valores nulos ─────────────────────────────────────────────────────────
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({'nulos': null_counts, 'porcentaje_%': null_pct})

print('=== Valores nulos por columna ===')
display(null_summary)

total_nulls = null_counts.sum()
print(f'\nTotal de valores nulos: {total_nulls}')
if total_nulls == 0:
    print('✓ El dataset no contiene valores nulos.')
else:
    print('⚠ Se encontraron valores nulos. Revisar antes de modelar.')

# ─── Timestamps duplicados ─────────────────────────────────────────────────
n_duplicates = df['timestamp'].duplicated().sum()
print(f'\n=== Timestamps duplicados: {n_duplicates} ===')
if n_duplicates == 0:
    print('✓ No hay timestamps duplicados.')
else:
    print('⚠ Hay timestamps duplicados. Esto puede romper operaciones de ventana temporal.')

# ─── Continuidad de la serie temporal ─────────────────────────────────────
expected_range = pd.date_range(start=df['timestamp'].min(),
                               end=df['timestamp'].max(),
                               freq='h')
missing_hours = expected_range.difference(df['timestamp'])
print(f'\n=== Horas faltantes en la serie temporal: {len(missing_hours)} ===')
if len(missing_hours) == 0:
    print('✓ La serie temporal es completamente continua.')
else:
    print(f'⚠ Faltan {len(missing_hours)} horas. Las primeras 5:')
    print(missing_hours[:5].tolist())

## 4. Distribución de la variable objetivo (`demand_kwh`)

Analizamos la distribución de la variable que queremos predecir. En problemas de regresión es importante entender:
- ¿Es simétrica o tiene sesgo?
- ¿Tiene múltiples modas (distribución bimodal)?
- ¿Cuál es el rango típico de valores?

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Histograma de demand_kwh', 'Box Plot de demand_kwh')
)

# Histograma
fig.add_trace(
    go.Histogram(
        x=df['demand_kwh'],
        nbinsx=80,
        name='demand_kwh',
        marker_color='steelblue',
        opacity=0.8
    ),
    row=1, col=1
)

# Box plot
fig.add_trace(
    go.Box(
        y=df['demand_kwh'],
        name='demand_kwh',
        marker_color='steelblue',
        boxpoints='outliers'
    ),
    row=1, col=2
)

fig.update_layout(
    title='Distribución de la Demanda Eléctrica',
    height=450,
    showlegend=False
)
fig.update_xaxes(title_text='demand_kwh', row=1, col=1)
fig.update_yaxes(title_text='Frecuencia', row=1, col=1)
fig.show()

# Estadísticas de la distribución
from scipy import stats as scipy_stats
skewness  = scipy_stats.skew(df['demand_kwh'])
kurtosis  = scipy_stats.kurtosis(df['demand_kwh'])

print(f'Media:    {df["demand_kwh"].mean():.2f} kWh')
print(f'Mediana:  {df["demand_kwh"].median():.2f} kWh')
print(f'Std:      {df["demand_kwh"].std():.2f} kWh')
print(f'Min:      {df["demand_kwh"].min():.2f} kWh')
print(f'Max:      {df["demand_kwh"].max():.2f} kWh')
print(f'Skewness: {skewness:.4f}  (>0 sesgado derecha, <0 sesgado izquierda)')
print(f'Kurtosis: {kurtosis:.4f}  (>0 más apuntada que normal)')

## 5. Serie temporal: demanda a lo largo del tiempo

Visualizamos la serie temporal completa para detectar tendencias, estacionalidades y anomalías.

- **Primer gráfico**: primeros 30 días → resolución horaria, patrón diario visible
- **Segundo gráfico**: serie completa de 3 años → estacionalidad anual visible

In [ ]:
# ─── Zoom: primeros 30 días ────────────────────────────────────────────────
df_30d = df.head(30 * 24)   # 30 días × 24 horas

fig1 = px.line(
    df_30d,
    x='timestamp',
    y='demand_kwh',
    title='Demanda Eléctrica — Primeros 30 días (resolución horaria)',
    labels={'demand_kwh': 'Demanda (kWh)', 'timestamp': 'Fecha'},
    color_discrete_sequence=['steelblue']
)
fig1.update_layout(height=380)
fig1.show()

# ─── Serie completa con media diaria ──────────────────────────────────────
df_daily = df.resample('D', on='timestamp')['demand_kwh'].agg(['mean', 'min', 'max']).reset_index()
df_daily.columns = ['timestamp', 'mean', 'min', 'max']

fig2 = go.Figure()

# Banda min-max
fig2.add_trace(go.Scatter(
    x=pd.concat([df_daily['timestamp'], df_daily['timestamp'][::-1]]),
    y=pd.concat([df_daily['max'], df_daily['min'][::-1]]),
    fill='toself',
    fillcolor='rgba(70, 130, 180, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Rango diario (min-max)'
))

# Media diaria
fig2.add_trace(go.Scatter(
    x=df_daily['timestamp'],
    y=df_daily['mean'],
    mode='lines',
    name='Media diaria',
    line=dict(color='steelblue', width=1.5)
))

fig2.update_layout(
    title='Demanda Eléctrica — Serie completa (3 años, media diaria)',
    xaxis_title='Fecha',
    yaxis_title='Demanda (kWh)',
    height=400,
    hovermode='x unified'
)
fig2.show()

print('Observaciones:')
print('  - Patrón diario claro: picos matutinos y vespertinos')
print('  - Patrón semanal: caída los fines de semana')
print('  - Patrón anual: picos en invierno y verano (temperatura extrema)')

## 6. Estacionalidades

La demanda eléctrica tiene múltiples niveles de estacionalidad superpuestos:

- **Hora del día**: picos en las mañanas y tardes-noches
- **Día de la semana**: industria y comercio bajan los fines de semana
- **Mes del año**: calefacción en invierno, aire acondicionado en verano

Visualizar estas estacionalidades nos ayudará a decidir qué features crear más adelante.

In [ ]:
import plotly.figure_factory as ff

# ─── Heatmap: hora × día de la semana ─────────────────────────────────────
pivot_hour_dow = (
    df.groupby(['hour', 'day_of_week'])['demand_kwh']
    .mean()
    .unstack('day_of_week')
    .round(1)
)
pivot_hour_dow.columns = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

fig_heatmap = px.imshow(
    pivot_hour_dow,
    labels=dict(x='Día de la semana', y='Hora del día', color='kWh'),
    title='Demanda media por Hora × Día de la semana',
    color_continuous_scale='Blues',
    aspect='auto'
)
fig_heatmap.update_layout(height=450)
fig_heatmap.show()

# ─── Boxplot por mes ───────────────────────────────────────────────────────
month_names = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
df['month_name'] = df['month'].apply(lambda m: month_names[m-1])

fig_box = px.box(
    df,
    x='month',
    y='demand_kwh',
    title='Distribución de Demanda por Mes',
    labels={'demand_kwh': 'Demanda (kWh)', 'month': 'Mes'},
    color='month',
    color_discrete_sequence=px.colors.sequential.Blues_r
)
fig_box.update_xaxes(tickvals=list(range(1,13)), ticktext=month_names)
fig_box.update_layout(height=450, showlegend=False)
fig_box.show()

print('Insights del heatmap:')
print('  - Máxima demanda: 18:00–21:00 de Lunes a Viernes')
print('  - Mínima demanda: 02:00–05:00 del Sábado y Domingo')
print('\nInsights del boxplot por mes:')
print('  - Meses de mayor demanda: Enero/Febrero (calefacción) y Julio/Agosto (AC)')
print('  - Meses de menor demanda: Abril/Mayo/Octubre (clima templado)')

## 7. Correlaciones

La matriz de correlación de Pearson nos muestra relaciones **lineales** entre variables numéricas.

Interpretación:
- Valor cercano a **+1**: correlación positiva fuerte
- Valor cercano a **-1**: correlación negativa fuerte
- Valor cercano a **0**: sin correlación lineal (puede haber relación no-lineal)

> **Importante**: la correlación de Pearson no captura relaciones cíclicas (ej. `hour` con `demand_kwh` es cíclica, no lineal). Por eso usaremos encodings sen/cos en el siguiente notebook.

In [ ]:
numeric_cols = ['hour', 'day_of_week', 'month', 'is_weekend',
                'is_holiday', 'temperature_c', 'demand_kwh']

corr_matrix = df[numeric_cols].corr().round(3)

fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Matriz de Correlación de Pearson',
    aspect='auto'
)
fig_corr.update_layout(height=500)
fig_corr.show()

# Correlaciones con la variable objetivo, ordenadas
print('\n=== Correlación con demand_kwh ===')
corr_target = corr_matrix['demand_kwh'].drop('demand_kwh').sort_values(key=abs, ascending=False)
for feat, val in corr_target.items():
    bar = '█' * int(abs(val) * 20)
    sign = '+' if val > 0 else '-'
    print(f'  {feat:<15} {sign}{abs(val):.3f}  {bar}')

## 8. Análisis de outliers

Los outliers pueden ser:
1. **Errores de medición**: valores imposibles físicamente
2. **Eventos reales**: picos de consumo por eventos especiales, fallos de red, etc.

Usamos el método **IQR (Rango Intercuartílico)**:
- `Q1` = percentil 25
- `Q3` = percentil 75
- `IQR = Q3 - Q1`
- Outliers: valores fuera de `[Q1 - 1.5×IQR, Q3 + 1.5×IQR]`

In [ ]:
# ─── Detección de outliers con IQR ────────────────────────────────────────
Q1  = df['demand_kwh'].quantile(0.25)
Q3  = df['demand_kwh'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers    = df[(df['demand_kwh'] < lower_bound) | (df['demand_kwh'] > upper_bound)]
outlier_pct = len(outliers) / len(df) * 100

print(f'Q1:          {Q1:.2f} kWh')
print(f'Q3:          {Q3:.2f} kWh')
print(f'IQR:         {IQR:.2f} kWh')
print(f'Límite inf:  {lower_bound:.2f} kWh')
print(f'Límite sup:  {upper_bound:.2f} kWh')
print(f'Outliers:    {len(outliers):,} ({outlier_pct:.2f}% del total)')

# Visualización
fig_out = go.Figure()

# Serie normal
mask_normal  = (df['demand_kwh'] >= lower_bound) & (df['demand_kwh'] <= upper_bound)
mask_outlier = ~mask_normal

fig_out.add_trace(go.Scatter(
    x=df.loc[mask_normal, 'timestamp'],
    y=df.loc[mask_normal, 'demand_kwh'],
    mode='markers',
    marker=dict(size=2, color='steelblue', opacity=0.4),
    name='Normal'
))

fig_out.add_trace(go.Scatter(
    x=df.loc[mask_outlier, 'timestamp'],
    y=df.loc[mask_outlier, 'demand_kwh'],
    mode='markers',
    marker=dict(size=6, color='crimson', symbol='x'),
    name='Outlier (IQR)'
))

# Líneas de límite
for bound, label in [(lower_bound, 'Límite inf'), (upper_bound, 'Límite sup')]:
    fig_out.add_hline(
        y=bound, line_dash='dash', line_color='orange',
        annotation_text=label, annotation_position='top right'
    )

fig_out.update_layout(
    title='Detección de Outliers — Método IQR',
    xaxis_title='Fecha',
    yaxis_title='Demanda (kWh)',
    height=420
)
fig_out.show()

if len(outliers) > 0:
    print('\nMuestra de outliers detectados:')
    display(outliers[['timestamp', 'hour', 'day_of_week', 'temperature_c', 'demand_kwh']].head(10))

## 9. Conclusiones

### Lo que aprendimos del dataset

**Estructura del dataset:**
- ~26,136 registros horarios (2022-2024, sin la primera semana por los lags)
- Sin valores nulos ni timestamps duplicados
- Serie temporal continua y limpia

**Distribución del target `demand_kwh`:**
- Distribución aproximadamente bimodal (horas de baja y alta demanda)
- Rango típico: ~835–1300 kWh (percentiles 5–95)
- Pocos outliers extremos (<1%), probablemente eventos reales

**Estacionalidades detectadas:**
| Ciclo | Descripción |
|-------|-------------|
| Diario | Picos a las 8-9h y 19-21h; mínimo a las 3-5h |
| Semanal | Fin de semana ~20% menos que días laborables |
| Anual | Máximos en enero-febrero y julio-agosto |

**Features más predictivos (correlación con `demand_kwh`):**
1. `temperature_c` — correlación cuadrática fuerte (efecto frío Y calor)
2. `is_weekend` — fuerte efecto negativo sobre la demanda
3. `hour` — correlación cíclica (necesita encoding sin/cos)
4. `month` — correlación cíclica con estacionalidad anual
5. `is_holiday` — efecto parecido al fin de semana

### Próximos pasos

En el siguiente notebook (`01_feature_engineering.ipynb`) transformaremos estas variables crudas en features listos para ML:
- Encodings cíclicos (sin/cos) para hora, día y mes
- Features de lag (demanda de hace 24h, 168h)
- Medias móviles
- Por qué el pipeline no escala las features (LightGBM es invariante a la escala) y cómo evitar el training-serving skew cuando sí se escala
- Split temporal correcto (¡sin data leakage!)